In [1]:
import langchain
langchain.__version__

'1.2.15'

### GROQ MODEL  INTEGRATION

In [23]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [25]:
from langchain.chat_models import init_chat_model
model = init_chat_model(
      "llama-3.1-8b-instant",  # Model ka naam jo Groq support karta hai
    model_provider="groq" # Ye batana zaroori hai ki hum Groq use kar rahe hain
)
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000140E7027B10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000140E7214550>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [26]:
# first generative ai application
response = model.invoke("write essay about ai")
print(response.content)

**The Rise of Artificial Intelligence: Opportunities and Challenges**

Artificial intelligence (AI) has been a subject of fascination and debate for decades. From its humble beginnings in the 1950s to the present day, AI has evolved into a powerful tool that is transforming industries, revolutionizing the way we live and work, and raising fundamental questions about the future of humanity.

**Defining AI**

Artificial intelligence refers to the development of computer systems that can perform tasks that would typically require human intelligence, such as learning, problem-solving, decision-making, and perception. AI systems can be trained on large datasets, enabling them to recognize patterns, identify relationships, and make predictions. These capabilities have led to the creation of AI-powered applications, including virtual assistants, language translation software, image recognition systems, and self-driving cars.

**Benefits of AI**

The benefits of AI are numerous and far-reachin

### GOOGLE- GEMINI Model integration

In [11]:
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

from langchain.chat_models import init_chat_model
model1 = init_chat_model("google_genai:gemini-2.5-flash-lite", temperature=0.9, max_tokens=20)

In [12]:
response = model1.invoke("write essay about ai")
print(response.content)

## The Algorithmic Dawn: Navigating the Promise and Peril of Artificial Intelligence

The 


In [13]:
for chunk in model1.stream("write essay about ai"):
    print(chunk.text, end="", flush=True)

## The Dawn of a New Intelligence: Navigating the Promise and Peril of Artificial Intelligence

We

In [14]:
responses = model.batch(
    [
        "why india is the best country in the world?",
        "why python is the best programming language in the world?"
        "why is the sky blue?"
    ]
)

for response in responses:
    print(response)

content='I must clarify that ranking countries as the "best" can be subjective and often depends on various criteria. India, like any other country, has its unique strengths and challenges. Here\'s a balanced view of some reasons why India is considered a great nation:\n\n1. **Rich Cultural Heritage**: India is home to a vibrant and diverse cultural heritage, with a history dating back over 5,000 years. From the Taj Mahal, a UNESCO World Heritage Site, to the sacred city of Varanasi, and the numerous festivals like Diwali and Holi, India has a rich cultural tapestry.\n2. **Diverse Geography**: India is a vast and geographically diverse country, with snow-capped mountains, serene beaches, and lush forests. The Himalayas, the world\'s highest mountain range, runs through the northern part of the country.\n3. **Economic Growth**: India has experienced rapid economic growth in recent years, with a GDP growth rate of around 7% in the past decade. The country is now the 5th largest economy i

In [37]:
from langchain.tools import tool
@tool
def get_weather(location: str) -> str:
    """get the weather at a location """
    return f"The weather at {location} is sunny with a high of 25°C."

model_with_tool = model.bind_tools([get_weather])

In [38]:
response = model_with_tool.invoke("what is the weather in delhi?")
print(response)

content='' additional_kwargs={'tool_calls': [{'id': 'asg06qj2m', 'function': {'arguments': '{"location":"delhi"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 220, 'total_tokens': 235, 'completion_time': 0.03504893, 'completion_tokens_details': None, 'prompt_time': 0.016643383, 'prompt_tokens_details': None, 'queue_time': 0.045322717, 'total_time': 0.051692313}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d59d6-cc4c-7993-b34e-6274bb2dbadd-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'delhi'}, 'id': 'asg06qj2m', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 220, 'output_tokens': 15, 'total_tokens': 235}


In [39]:
response.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'delhi'},
  'id': 'asg06qj2m',
  'type': 'tool_call'}]

### Tools Execurion Loop

In [44]:
from langchain_core.messages import ToolMessage

messages = [{"role": "user", "content": "What is the weather in delhi?"}]
ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    # 1. Execute the tool
    tool_output = get_weather.invoke(tool_call["args"])
    tool_message = ToolMessage(
        content=str(tool_output), 
        tool_call_id=tool_call["id"]
    )
    messages.append(tool_message)

# 3. Invoke the model again with the tool's result included in history
final_response = model_with_tool.invoke(messages)

# 4. Use .content instead of .text
print(final_response.content)

In [45]:
messages

[{'role': 'user', 'content': 'What is the weather in delhi?'},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '1x2swa7sv', 'function': {'arguments': '{"location":"delhi"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 220, 'total_tokens': 235, 'completion_time': 0.027834877, 'completion_tokens_details': None, 'prompt_time': 0.013792656, 'prompt_tokens_details': None, 'queue_time': 0.046211203, 'total_time': 0.041627533}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d59db-6867-7c70-bacd-d19ee431ac72-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'delhi'}, 'id': '1x2swa7sv', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 220, 'output_tokens': 15, 'total_tokens': 235}),
 ToolMessage(content='The weath